In [9]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict , Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import MemorySaver

In [10]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict): 
    messages: Annotated[list[BaseMessage], add_messages]
    

In [11]:
llm = ChatOllama(model='llama3.2')

In [12]:
def chat_node(state: ChatState): 
    #take a user query from state
    messages = state['messages']

    # send to llm
    response = llm.invoke(messages) 

    # response store state
    return {"messages": response}

In [16]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

#add nodes
graph.add_node("chat_node", chat_node)

#add edges 
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)
workflow = graph.compile(checkpointer=checkpointer)

In [17]:
inital_state = {
    "messages": [HumanMessage(content="What is the capital of Nepal?")]
}
workflow.invoke(inital_state)['messages'][-1].content

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [18]:
thread_id = '1'
while True:
    user_message = input("type : ")
    print("User: ", user_message)

    if user_message.strip().lower() in ['exit', 'quit', 'bye', '0']:
        break
    config = {'configurable': {'thread_id': thread_id}}
    response = workflow.invoke({

        'messages': [HumanMessage(content=user_message)]
    }, config=config)

    print('AI: ', response['messages'][-1].content)

User:  Hi my name is milan
AI:  Hello Milan! It's nice to meet you. Is there something I can help you with or would you like to chat?
User:  Can you tell my name
AI:  Your name is Milan. Is there anything else you'd like to talk about?
User:  can you add 10 and 110
AI:  The result of adding 10 and 110 is 120.
User:  then add 20
AI:  120 + 20 = 140
User:  exit


In [19]:
workflow.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi my name is milan', additional_kwargs={}, response_metadata={}, id='2c15b2ac-8c20-4d3a-9e92-2c21188c10a0'), AIMessage(content="Hello Milan! It's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-10-30T13:49:52.8570933Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7283456900, 'load_duration': 3907492500, 'prompt_eval_count': 30, 'prompt_eval_duration': 402392000, 'eval_count': 26, 'eval_duration': 2963653000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--01a18d12-dae4-756b-a166-0858bac2f9ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 26, 'total_tokens': 56}), HumanMessage(content='Can you tell my name', additional_kwargs={}, response_metadata={}, id='3faf6994-bdb1-4d5d-aca8-e7f5126c7e56'), AIMessage(content="Your